## Punjabi Speech-to-Text Transcription with Gemma 3n E2B IT

Google Colab notebook — run on a **T4 GPU** runtime.

---
### Prerequisites
1. Accept the [Gemma license](https://huggingface.co/google/gemma-3n-e2b-it) on Hugging Face.
2. Create an [HF access token](https://huggingface.co/settings/tokens).
3. In Colab: **Secrets** (🔑) → add  with your token.

In [ ]:
# @title Install dependencies
import subprocess, sys, importlib.metadata, warnings
warnings.filterwarnings("ignore")

REQUIRED = {
    "transformers": "4.53.0",
    "accelerate": None,
    "torch": None,
    "librosa": None,
    "soundfile": None,
}


def _ensure_deps():
    for pkg, min_ver in REQUIRED.items():
        needs_upgrade = False
        try:
            ver = importlib.metadata.version(pkg)
            if min_ver:
                parts = tuple(int(x) for x in ver.split("."))
                req = tuple(int(x) for x in min_ver.split("."))
                if parts < req:
                    needs_upgrade = True
        except importlib.metadata.PackageNotFoundError:
            needs_upgrade = True
        if needs_upgrade:
            spec = f"{pkg}>={min_ver}" if min_ver else pkg
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "--upgrade", spec]
            )

_ensure_deps()
print("All dependencies satisfied.")

In [2]:
# @title Imports & Device Detection
import torch
import librosa
import numpy as np
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE.startswith("cuda") else torch.float32
print(f"Device: {DEVICE}  |  dtype: {DTYPE}")

Device: cuda:0  |  dtype: torch.bfloat16


In [5]:
# @title Load HF Token & Model
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except (ImportError, ValueError):
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Set it in Colab Secrets (key: HF_TOKEN) "
        "or as an environment variable."
    )

MODEL_ID = "google/gemma-3n-e4b-it"

try:
    print("Loading processor...")
    processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

    print("Loading model...")
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=DTYPE,
        device_map="auto",
        attn_implementation="eager",
    )
    model.eval()
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("\nTIP: Ensure you have accepted the license at https://huggingface.co/google/gemma-3n-e2b-it and your HF_TOKEN is valid.")

Loading processor...
Loading model...


Loading weights:   0%|          | 0/1526 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded successfully.


In [6]:
# @title Audio Preprocessing
MAX_AUDIO_SEC = 30


def prepare_audio(file_path: str) -> np.ndarray:
    """Load & preprocess audio for Gemma 3n.

    Returns a 1-D float32 NumPy array resampled to 16 kHz, mono,
    normalised to [-1, 1].
    """
    if not file_path:
        raise ValueError("file_path must be a non-empty string.")

    audio, sr = librosa.load(file_path, sr=16000, mono=True)

    duration = audio.shape[0] / sr
    if duration > MAX_AUDIO_SEC:
        print(
            f"WARNING: Audio is {duration:.1f}s exceeds {MAX_AUDIO_SEC}s. "
            "Gemma 3n native clip threshold may degrade quality."
        )

    return audio

In [26]:
# @title Transcription Function with Real-time Output & Repetition Penalty

ASR_PROMPT = (
    "Transcribe the following speech segment in Punjabi into "
    "Gurmukhi text. Output only the raw transcript, with no "
    "introductory text or newlines."
)

def transcribe(file_path: str, max_new_tokens: int = 448) -> str:
    """Transcribes long audio by splitting it into ~30s chunks and prints real-time updates."""
    import librosa
    import numpy as np
    import torch
    from IPython.display import display, Markdown

    # 1. Load the full audio
    audio, sr = librosa.load(file_path, sr=16000, mono=True)
    total_samples = len(audio)
    duration = total_samples / sr

    target_len_samples = 30 * sr
    chunks = []

    # 2. Split logic
    start = 0
    while start < total_samples:
        end = min(start + target_len_samples, total_samples)
        if end < total_samples:
            search_start = max(start, end - 5 * sr)
            search_range = audio[search_start : end]
            rms = librosa.feature.rms(y=search_range, frame_length=2048, hop_length=512)[0]
            min_rms_idx = np.argmin(rms)
            silence_offset = (min_rms_idx * 512)
            end = search_start + silence_offset
            if end <= start:
                end = min(start + target_len_samples, total_samples)
        chunks.append(audio[start:end])
        start = end

    print(f"Audio duration: {duration:.2f}s | Split into {len(chunks)} chunks.")

    full_transcript = []

    # 3. Process each chunk sequentially
    try:
        for i, chunk_data in enumerate(chunks):
            print(f"--- Processing chunk {i+1}/{len(chunks)} ---")

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "audio", "audio": chunk_data},
                        {"type": "text", "text": ASR_PROMPT},
                    ],
                },
            ]

            inputs = processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            input_len = inputs["input_ids"].shape[-1]

            with torch.inference_mode():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.1,  # Added to prevent loops
                )

            response_ids = generated_ids[0][input_len:]
            chunk_text = processor.decode(response_ids, skip_special_tokens=True).strip()

            # Immediate Output
            print(f"Chunk {i+1} Transcript: {chunk_text}")
            full_transcript.append(chunk_text)
    finally:
        print("\nTranscription step finished (completed or interrupted).")

    return " ".join(full_transcript)

In [23]:
# @title Run Transcription
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

AUDIO_PATH = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/AUD-20260402-WA0018.m4a"

try:
    print(f"Processing: {AUDIO_PATH}")
    # The function now prints and saves each chunk's transcript in real-time
    result = transcribe(AUDIO_PATH)
    print("\n--- FINAL FULL TRANSCRIPT (Gurmukhi) ---")
    print(result)
except Exception as exc:
    print(f"ERROR: {exc}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing: /content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/AUD-20260402-WA0018.m4a
Audio duration: 883.35s | Split into 33 chunks.
--- Processing chunk 1/33 ---
Chunk 1 Transcript: ਹਾਂ ਜੀ ਸਤਿ ਸ਼੍ਰੀ ਅਕਾਲ ਜੀ। ਮੇਰਾ ਨਾਮ ਅਮਰਤਪ੍ਰੀਤ ਸਿੰਘ ਜੀ ਪਿੰਡ ਮੇਰਾ ਮਨਸੂਹਾ ਕਲਾ ਜੀ ਤੇ ਪਿੰਡ ਮੇਰਾ ਮਨਸੂਹਾ ਕਲਾ ਜੀ। ਮਨਸੂਹਾ। ਹਾਂ ਜੀ। ਤੇ ਮੈਂ ਰੋਪੜ ਆਈ.ਆਈ.ਟੀ. ਵਿੱਚ ਜੋਬ ਕਰਦਾ ਜੀ। ਸਾਡਾ ਆਈ.ਆਈ.ਟੀ. ਜੀ. ਡਿਪਾਰਟਮੈਂਟ ਆ ਜੀ ਅਨਮੋਦ.ਆਈ.ਆਈ. ਠੀਕ ਜੀ ਅਨਮਿਕ ਆਪਣਾ ਸੰਸਕ੍ਰਿਤ ਭਾਸ਼ਾ ਦਾ ਬੈਲਡ ਹੈ ਜਿਵੇਂ ਕਿ ਅਨਨ ਭੋਜਨ।
--- Processing chunk 2/33 ---
Chunk 2 Transcript: ਜੋ ਅਸੀਂ ਖਾਂਦੇ ਹਾਂ ਜੀ। ਠੀਕ ਹੈ ਜੀ। AI ਜਿਹੜੀ ਹੁਣ ਟੈਕਨੋਲੋਜੀ ਚੱਲ ਰਹੀ ਹੈ। ਠੀਕ ਹੈ ਜੀ। ਸਾਡੇ ਕੋਲ ਟੈਕਨੋਲੋਜੀ ਹੈ, ਤੁਹਾਡੇ ਕੋਲ ਖੇਤੀਬਾੜੀ ਦਾ ਤਜਰਬਾ। ਇਹਨਾਂ ਦੋਨਾਂ ਨੂੰ ਜੋੜ ਕੇ ਜੀ ਆਪਾਂ ਇੱਕ ਐਪ ਤਿਆਰ ਕਰ ਰਹੇ ਹਾਂ ਜਿਸ ਦਾ ਨਾਮ ਹੈ ਅਜਰ ਸਖਾ। ਅਜਰ ਸਖਾ ਮੇਨ ਸਿਕਸ ਕਿਸਾਨਾਂ ਦਾ ਮਿੱਤਰ। ਸਾਡੀ ਐਪ ਦਾ ਜੋ ਐਪ ਤਿਆਰ ਕੀਤੀ ਹੈ ਜੀ ਅਸੀਂ ਬਹੁਤ ਬਹੁਤ ਧੰਨਵਾਦ ਕਰਦੇ ਹਾ

In [15]:
!pip install -q indic-transliteration

In [25]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def gurmukhi_to_devanagari(text):
    return transliterate(text, sanscript.GURMUKHI, sanscript.DEVANAGARI)

if 'result' in locals() and result:
    # Transliterating the current content of 'result'
    devanagari_transcript = gurmukhi_to_devanagari(result)
    print("--- CURRENT TRANSCRIPT (Devanagari Transliteration) ---")
    print(devanagari_transcript)
else:
    print("No transcript found to transliterate.")

--- CURRENT TRANSCRIPT (Devanagari Transliteration) ---
हां जी सति स਼्री अकाल जी। मेरा नाम अमरतप्रीत सिंघ जी पिंड मेरा मनसूहा कला जी ते पिंड मेरा मनसूहा कला जी। मनसूहा। हां जी। ते मैं रोपड़ आई.आई.टी. विच्च जोब करदा जी। साडा आई.आई.टी. जी. डिपारटमैंट आ जी अनमोद.आई.आई. ठीक जी अनमिक आपणा संसक्रित भास਼ा दा बैलड है जिवें कि अनन भोजन। जो असीं खांदे हां जी। ठीक है जी। AI जिहड़ी हुण टैकनोलोजी चल्ल रही है। ठीक है जी। साडे कोल टैकनोलोजी है, तुहाडे कोल खेतीबाड़ी दा तजरबा। इहनां दोनां नूं जोड़ के जी आपां इक्क ऐप तिआर कर रहे हां जिस दा नाम है अजर सखा। अजर सखा मेन सिकस किसानां दा मित्तर। साडी ऐप दा जो ऐप तिआर कीती है जी असीं बहुत बहुत धंनवाद करदे हां AI दे प्रोफैसरां दा जिहनां ने साडी खेतीबाड़ी नूं वी इक्क टैकनोलोजी दे नाल जोड़न दी कोस਼िस਼ कीती है जी। देखिआ जाब जी अज्ज दे अधारत हर इक्क मतलक कि चीज਼ ने अह अग्गे अपडेट लै लई है जी हनां। मतलब कि हर इक्क चीज਼ टैकनोलोजी नाल जुड़ गई है जी। जिवें साडे बज਼ुरगां ने देखिआ जी पहिलां मालो कि किसे नाल गल्लबात करनी हुंदी सी चिੱक्खिआ नहीं गल्लबात करदे। 10-15 दिनां उहदा जवा

In [27]:
import os

# @title Save Results to Drive
SAVE_DIRECTORY = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Transcripts/Gemma3n-e4b/FullNarration_Gagg"

if not os.path.exists(SAVE_DIRECTORY):
    os.makedirs(SAVE_DIRECTORY)

if 'result' in locals() and result:
    gurmukhi_file = os.path.join(SAVE_DIRECTORY, "transcript_punjabi.txt")
    with open(gurmukhi_file, "w", encoding="utf-8") as f:
        f.write(result)
    print(f"Saved Gurmukhi transcript to: {gurmukhi_file}")

if 'devanagari_transcript' in locals() and devanagari_transcript:
    devanagari_file = os.path.join(SAVE_DIRECTORY, "transcript_devanagari.txt")
    with open(devanagari_file, "w", encoding="utf-8") as f:
        f.write(devanagari_transcript)
    print(f"Saved Devanagari transliteration to: {devanagari_file}")

Saved Gurmukhi transcript to: /content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Transcripts/transcript_punjabi.txt
Saved Devanagari transliteration to: /content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Transcripts/transcript_devanagari.txt
